# 2. Primeira simulação: antena patch no ar

Neste notebook você usa o openEMS diretamente, passo a passo, como no tutorial oficial
(`Simple_Patch_Antenna.py`). Nos próximos notebooks, o pacote `emslab` faz estes passos por você.

**Método FDTD em uma frase:** o openEMS divide o espaço numa malha de pequenas caixas e calcula
os campos elétrico e magnético em cada caixa, passo a passo no tempo. Depois, a transformada
de Fourier dá a resposta em frequência (por exemplo, o S11).

Leia [docs/como_funciona.md](https://github.com/prof-davifr/openEMS-win/blob/main/docs/como_funciona.md)
para uma explicação curta.

## 2.1 Dimensões da antena

As fórmulas clássicas (Balanis, cap. 14) dão as dimensões iniciais de um patch em FR-4
(εr = 4,4; tan δ = 0,02; 1,6 mm) para 3,5 GHz.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from CSXCAD import ContinuousStructure
from openEMS import openEMS
from openEMS.physical_constants import C0, EPS0
import emslab

projeto = emslab.patch_retangular(f_ghz=3.5, er=4.4, h=1.6, tand=0.02)
print(projeto)

## 2.2 Parâmetros da simulação

Mude os valores desta célula para fazer experimentos. Unidades: mm e Hz.

In [ ]:
L = projeto.comprimento      # comprimento do patch (direção x): define a ressonância
W = projeto.largura          # largura do patch (direção y)
x_sonda = -projeto.alimentacao   # posição da sonda de 50 ohm
h = projeto.substrato_h
er = projeto.substrato_er
tand = projeto.substrato_tand
lado = projeto.substrato_lado    # substrato e plano de terra quadrados

f_min, f_max = 2e9, 5e9          # banda de interesse
f0, fc = (f_min + f_max) / 2, (f_max - f_min) / 2

## 2.3 Motor FDTD, excitação e contornos

- A excitação é um pulso gaussiano com energia de `f_min` a `f_max`.
- `MUR` nos seis lados: as ondas saem da caixa sem refletir (quase).
- A simulação para quando a energia cai 40 dB (`EndCriteria=1e-4`).

In [ ]:
fdtd = openEMS(NrTS=30000, EndCriteria=1e-4)
fdtd.SetGaussExcite(f0, fc)
fdtd.SetBoundaryCond(["MUR"] * 6)

csx = ContinuousStructure()
fdtd.SetCSX(csx)
malha = csx.GetGrid()
malha.SetDeltaUnit(1e-3)                          # 1 unidade da malha = 1 mm

res = C0 / f_max / 1e-3 / 15 / np.sqrt(er)        # ~15 células por comprimento de onda no FR-4
caixa = lado / 2 + 40                             # 40 mm de ar em volta da antena
malha.AddLine("x", [-caixa, caixa])
malha.AddLine("y", [-caixa, caixa])
malha.AddLine("z", [-40, h + 40])

## 2.4 Geometria: substrato, plano de terra, patch e porta

In [ ]:
kappa = 2 * np.pi * f0 * EPS0 * er * tand          # perda dielétrica como condutividade (S/m)
substrato = csx.AddMaterial("substrato", epsilon=er, kappa=kappa)
substrato.AddBox([-lado / 2, -lado / 2, 0], [lado / 2, lado / 2, h], priority=0)
malha.AddLine("z", np.linspace(0, h, 5))          # 4 células na espessura do substrato

terra = csx.AddMetal("plano_de_terra")            # condutor perfeito (PEC)
terra.AddBox([-lado / 2, -lado / 2, 0], [lado / 2, lado / 2, 0], priority=10)
fdtd.AddEdges2Grid(dirs="xy", properties=terra)

patch = csx.AddMetal("patch")
patch.AddBox([-L / 2, -W / 2, h], [L / 2, W / 2, h], priority=10)
fdtd.AddEdges2Grid(dirs="xy", properties=patch, metal_edge_res=res / 2)   # regra 1/3-2/3

porta = fdtd.AddLumpedPort(1, 50, [x_sonda, 0, 0], [x_sonda, 0, h], "z", 1.0, priority=5, edges2grid="xy")

malha.SmoothMeshLines("all", res, 1.4)
for eixo in "xyz":   # tira erros de arredondamento da malha (uma folha de metal fora da linha some)
    malha.SetLines(eixo, np.unique(np.round(malha.GetLines(eixo), 6)))
print("Linhas da malha (x, y, z):", [len(malha.GetLines(e)) for e in "xyz"])

## 2.5 Ver a geometria em 3D

Esta célula abre o **AppCSXCAD** numa janela separada. Gire com o mouse e confira a antena.
Feche a janela quando terminar. A simulação não precisa desta etapa.

In [ ]:
emslab.ver_geometria(csx, "patch_no_ar")

## 2.6 Rodar a simulação

Num PC comum, demora de 10 s a 2 min. As mensagens do openEMS aparecem na janela preta do Jupyter, não aqui.

In [ ]:
import os, time

pasta = emslab.pasta_simulacao("patch_no_ar")
pasta_do_notebook = os.getcwd()
inicio = time.time()
fdtd.Run(str(pasta), cleanup=True)
os.chdir(pasta_do_notebook)    # o openEMS muda a pasta atual; voltar para a pasta do notebook
print(f"Simulação terminou em {time.time() - inicio:.0f} s")

## 2.7 Resultado: S11 e impedância de entrada

**S11** é a fração da onda que volta para o cabo. Na ressonância, a antena aceita a energia e o |S11| cai.
Abaixo de −10 dB, menos de 10 % da potência volta.

In [ ]:
f = np.linspace(f_min, f_max, 1001)
porta.CalcPort(str(pasta), f)
s11 = porta.uf_ref / porta.uf_inc
zin = porta.uf_tot / porta.if_tot
s11_db = 20 * np.log10(np.abs(s11))

i = np.argmin(s11_db)
print(f"Ressonância simulada: {f[i] / 1e9:.3f} GHz (projeto: {projeto.f_projeto} GHz), |S11| = {s11_db[i]:.1f} dB")
print(f"Zin na ressonância: {zin[i].real:.1f} {zin[i].imag:+.1f}j ohm")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), tight_layout=True)
ax1.plot(f / 1e9, s11_db, "k-", linewidth=2)
ax1.axhline(-10, color="gray", linestyle=":")
ax1.set_xlabel("Frequência (GHz)"); ax1.set_ylabel("|S11| (dB)"); ax1.set_title("Coeficiente de reflexão"); ax1.grid(True)
ax2.plot(f / 1e9, zin.real, "k-", label="Re{Zin}")
ax2.plot(f / 1e9, zin.imag, "r--", label="Im{Zin}")
ax2.set_xlabel("Frequência (GHz)"); ax2.set_ylabel("Zin (ohm)"); ax2.set_title("Impedância de entrada"); ax2.grid(True); ax2.legend()

## 2.8 Exercícios

1. A ressonância simulada fica um pouco abaixo de 3,5 GHz. Diminua `L` (célula 2.2) para chegar a 3,5 GHz.
   Rode as células de 2.2 até 2.7 de novo.
2. Mude `x_sonda` para 0 (sonda no centro). O que acontece com o |S11|? Por quê?
3. Mude `tand` para 0,001 (substrato de baixa perda, como o Rogers). O que muda?